<a href="https://colab.research.google.com/github/zmhibner-gif/ml_internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zmhibner-gif/ml_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:

from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

# Main warehouse path
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

print("Hugging Face connection ready.")

NotebookAccessError: Notebook does not have access to secret HF_TOKEN

In [ ]:
# march slice data set set up from a previous notebook
MONTH = "2026-03"

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{WAREHOUSE}/fact_content_daily_performance"
MAR = f"read_parquet('{FACT}/month={MONTH}/*.parquet')"

feature_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,

        100.0 * SUM(gsc_clicks)
        / NULLIF(SUM(gsc_impressions), 0) AS ctr,

        SUM(gsc_sum_position)
        / NULLIF(SUM(gsc_impressions), 0) AS avg_position,

        COUNT(*) FILTER (
            WHERE gsc_impressions > 0
        ) AS days_with_impressions,

        COUNT(*) FILTER (
            WHERE gsc_clicks > 0
        ) AS days_with_clicks

    FROM {MAR}
    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) > 0
""").df()

print(feature_df.shape[0], "pages,", feature_df.shape[1], "columns")
feature_df.head(10)

## 1. My rule and its reason codes

**Rule:**
Prioritize pages that already receive meaningful search visibility but have a lower-than-expected CTR for their average search position. These pages may represent opportunities where the search result is being seen but is not attracting as many clicks as expected.

**Reason code:**
`LOW_CTR_FOR_POSITION`

This reason code is assigned to pages that meet the visibility requirement and have weak CTR relative to their search position.

#1.2 Signal checks

In [ ]:
# Make a separate copy of the March dataset for the signal checks.
signal_df = feature_df.copy()

import pandas as pd


In [ ]:
## 1. Signal check: Position/CTR
# Group pages by their average Google search position.
position_bins = [0, 3, 5, 10, 20, 50, float("inf")]
position_labels = ["1-3", "4-5", "6-10", "11-20", "21-50", "51+"]

signal_df["position_bucket"] = pd.cut(
    signal_df["avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

# Calculate the number of pages and typical CTR in each position group.
position_table = (
    signal_df.groupby("position_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        mean_ctr=("ctr", "mean"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

position_table

**Signal 1 verdict: CONFIRMED**

Mean CTR decreases as average search position gets worse, from about 1.17% for positions 1–3 to 0.08% for positions 51+. This supports using CTR together with average position in the baseline rule.

In [ ]:
## 2. Signal Check: Impression volume
# Group pages by how many impressions they received.
impression_bins = [0, 10, 50, 100, 500, 1000, 5000, float("inf")]

impression_labels = [
    "1-10",
    "11-50",
    "51-100",
    "101-500",
    "501-1,000",
    "1,001-5,000",
    "5,001+"
]

signal_df["impression_bucket"] = pd.cut(
    signal_df["impressions"],
    bins=impression_bins,
    labels=impression_labels,
    include_lowest=True
)

# Mark whether each page received at least one click during March.
signal_df["has_clicks"] = signal_df["days_with_clicks"] > 0

# Compare click activity across the different impression groups.
volume_table = (
    signal_df.groupby("impression_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_days_with_clicks=("days_with_clicks", "median"),
        share_with_clicks=("has_clicks", "mean")
    )
    .reset_index()
)

# Convert the share into a percentage to make the table easier to read.
volume_table["share_with_clicks"] = (
    volume_table["share_with_clicks"] * 100
).round(1)

volume_table

**Signal 2 verdict: CONFIRMED**

Pages with more impressions are much more likely to receive clicks. The share of pages with clicks rises from 3.1% in the lowest impression bucket to 98.2% in the highest. This supports using a minimum impression level in the baseline rule.


## 2. Build the ranked queue (writes the CSV)



In [ ]:
import os

# Add the average CTR for each position bucket to every page.
expected_ctr = position_table[
    ["position_bucket", "mean_ctr"]
].rename(columns={"mean_ctr": "expected_ctr"})

queue_df = signal_df.merge(
    expected_ctr,
    on="position_bucket",
    how="left"
)

# Calculate how far each page's CTR is below the average for its position bucket.
queue_df["ctr_gap"] = (
    queue_df["expected_ctr"] - queue_df["ctr"]
).clip(lower=0)

# Score the opportunity by combining the CTR gap with impression volume.
queue_df["baseline_score"] = (
    queue_df["impressions"] * queue_df["ctr_gap"] / 100
).round(2)

# Keep pages with at least 500 impressions and a positive CTR gap.
queue_df = queue_df[
    (queue_df["impressions"] >= 500) &
    (queue_df["baseline_score"] > 0)
].copy()

# Assign one reason code and one action label.
queue_df["reason_code"] = "LOW_CTR_FOR_POSITION"
queue_df["action"] = "REVIEW_LOW_CTR_PAGE"

# Rank pages from highest to lowest score.
queue_df = queue_df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

queue_df["rank"] = queue_df.index + 1

# Create the output folder if it does not already exist.
os.makedirs("work/outputs", exist_ok=True)

# Save the ranked queue required by the assignment.
output_path = "work/outputs/baseline_action_score.csv"
queue_df.to_csv(output_path, index=False)

print("Rows in ranked queue:", len(queue_df))
print("Saved:", output_path)

# Show the ten highest-ranked pages.
queue_df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "ctr",
        "avg_position",
        "expected_ctr",
        "baseline_score",
        "reason_code",
        "action"
    ]
].head(10)


## 3. Top 10 review

1. `content_44f34c0a90047651` — **Review low-CTR page:** very high impressions and CTR far below the average for its position; wrong if the page answers the search directly without requiring a click.
2. `content_8d7d99f109e19aa2` — **Review low-CTR page:** high visibility and CTR well below its position average; wrong if its queries naturally have a lower CTR.
3. `content_0e03de7680314cd5` — **Review low-CTR page:** very high impressions combined with a large CTR gap; wrong if it ranks for many low-intent or unrelated queries.
4. `content_4ffe18112a5642e3` — **Review low-CTR page:** high impression volume and below-expected CTR; wrong if ads or other search features are taking clicks away.
5. `content_8e1334d6356668e3` — **Review low-CTR page:** many impressions but almost no clicks despite a strong position; wrong if its query mix is very different from other pages in the same position bucket.
6. `content_eadb33b5df496f4a` — **Review low-CTR page:** extremely high impression volume makes even a smaller CTR gap important; wrong if its current CTR is normal for the specific queries it ranks for.
7. `content_fec55986a1868d62` — **Review low-CTR page:** high visibility with almost no clicks; wrong if its unusually low average-position value needs further checking.
8. `content_ec2e0346994fb5a5` — **Review low-CTR page:** high impressions and CTR below the position average; wrong if its CTR is normal for the specific searches where it appears.
9. `content_545bb6cc7081ded3` — **Review low-CTR page:** substantial visibility and a clear CTR gap; wrong if the position bucket is too broad to represent this page accurately.
10. `content_9c057b66c30a3abb` — **Review low-CTR page:** many impressions but almost no clicks; wrong if its unusually low average position makes the comparison unreliable.


## 4. Weak picks + leakage check



In [ ]:
# Show two questionable top-ranked pages for a closer look.
queue_df.loc[
    queue_df["rank"].isin([7, 10]),
    [
        "rank",
        "content_hash_id",
        "impressions",
        "ctr",
        "avg_position",
        "expected_ctr",
        "baseline_score"
    ]
]

# Confirm which inputs and time window were used by the baseline.
baseline_inputs = ["impressions", "ctr", "avg_position"]

print("Baseline inputs:", baseline_inputs)
print("Development window:", MONTH)


**Weak picks:** Ranks 7 and 10 look less reliable because their average-position values are unusually low. Their recommendations should be checked before taking action. Other pages may also have low CTR for valid reasons, such as query intent or search-result features.

**Leakage check:** The baseline uses only March 2026 impressions, CTR, and average position. No product flags, future-month data, or label-derived variables are used. Client and content IDs are only used to identify pages and are not part of the score.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.